In [3]:
# Required libraries
from pathlib import Path
from urllib.parse import urlparse
import pandas as pd


### Load the CSV of URLs extracted from the sitemap XML files.

In [4]:
# Define input and output paths
INPUT_CSV = Path("datacentermap_all_urls.csv")  # change if needed
OUTPUT_ALL_DETAIL = Path("02_datacentermap_detail_pages_global.csv")
OUTPUT_EU_CSV = Path("03_european_datacenters.csv")


In [5]:
# Load the list of URLs into a pandas DataFrame
df = pd.read_csv(INPUT_CSV, encoding="utf-8")

# Accept either of the two common column names
if "url" in df.columns:
    url_col = "url"
elif "Datacenter URL" in df.columns:
    url_col = "Datacenter URL"
else:
    raise ValueError("Could not find a URL column. Expected 'url' or 'Datacenter URL'.")

df = df[[url_col]].rename(columns={url_col: "Datacenter URL"}).dropna().drop_duplicates()
print(f"Loaded {len(df):,} unique URLs from {INPUT_CSV}")
df.head()


Loaded 25,734 unique URLs from datacentermap_all_urls.csv


,Datacenter URL
0,https://www.datacentermap.com/
1,https://www.datacentermap.com/about/
2,https://www.datacentermap.com/advertising/
3,https://www.datacentermap.com/afghanistan/
4,https://www.datacentermap.com/afghanistan/kabul/


### Normalize URL paths and compute path segments.

In [6]:
# Normalize URL paths
df["path"] = df["Datacenter URL"].astype(str).apply(lambda u: urlparse(u).path.strip("/"))
df = df[df["path"] != ""].copy()

# Split into segments
df["segments"] = df["path"].apply(lambda p: [seg for seg in p.split("/") if seg])
df["depth"] = df["segments"].apply(len)

# Keep a few segment columns for debugging
for i in range(6):
    df[f"seg{i}"] = df["segments"].apply(lambda s, i=i: s[i] if len(s) > i else None)

print("Depth distribution:")
display(df["depth"].value_counts().sort_index())
df.head()


Depth distribution:


depth
1      199
2    11029
3     8932
4     5456
5      117
Name: count, dtype: int64

,Datacenter URL,path,segments,depth,seg0,seg1,seg2,seg3,seg4,seg5
1,https://www.datacentermap.com/about/,about,[about],1,about,NaN,NaN,NaN,NaN,None
2,https://www.datacentermap.com/advertising/,advertising,[advertising],1,advertising,NaN,NaN,NaN,NaN,None
3,https://www.datacentermap.com/afghanistan/,afghanistan,[afghanistan],1,afghanistan,NaN,NaN,NaN,NaN,None
4,https://www.datacentermap.com/afghanistan/kabul/,afghanistan/kabul,"[afghanistan, kabul]",2,afghanistan,kabul,NaN,NaN,NaN,None
5,https://www.datacentermap.com/afghanistan/kabu...,afghanistan/kabul/acg-data-center,"[afghanistan, kabul, acg-data-center]",3,afghanistan,kabul,acg-data-center,NaN,NaN,None


### Exclude obvious non-facility branches and identify likely data center detail pages.

The key change versus the earlier notebook is that this logic **does not assume a fixed URL depth** such as `depth == 4`. Instead, it keeps likely **leaf pages** in the geographic URL tree, which is much more robust for non-US countries.

In [7]:
# Exclude obvious non-facility branches
EXCLUDED_FIRST_SEGMENTS = {
    "cloud",
    "providers",
    "markets",
    "internet-exchanges",
    "carrier-hotels",
    "about",
    "contact",
    "news",
    "blog",
    "sitemap",
    "legal",
    "privacy-policy",
    "terms",
}

geo_df = df[~df["seg0"].isin(EXCLUDED_FIRST_SEGMENTS)].copy()

# Build a set of all normalized paths
all_paths = set(geo_df["path"])

def is_leaf(path: str) -> bool:
    """A leaf page has no child URLs underneath it in the sitemap path tree."""
    prefix = path + "/"
    for other in all_paths:
        if other != path and other.startswith(prefix):
            return False
    return True

geo_df["is_leaf"] = geo_df["path"].apply(is_leaf)

# Keep likely detail pages:
# - leaf pages in the path tree
# - at least 3 segments to exclude top-level category pages
detail_df = geo_df[(geo_df["is_leaf"]) & (geo_df["depth"] >= 3)].copy()

print(f"Likely detail pages: {len(detail_df):,}")
display(detail_df[["Datacenter URL", "depth", "seg0", "seg1", "seg2", "seg3"]].head(20))


Likely detail pages: 13,518


,Datacenter URL,depth,seg0,seg1,seg2,seg3
5,https://www.datacentermap.com/afghanistan/kabu...,3,afghanistan,kabul,acg-data-center,NaN
6,https://www.datacentermap.com/afghanistan/kabu...,3,afghanistan,kabul,alef-technology,NaN
10,https://www.datacentermap.com/albania/tirana/a...,3,albania,tirana,albania-data-center-adc,NaN
11,https://www.datacentermap.com/albania/tirana/h...,3,albania,tirana,hostal-shpk,NaN
12,https://www.datacentermap.com/albania/tirana/k...,3,albania,tirana,keminet-shpk,NaN
13,https://www.datacentermap.com/albania/tirana/p...,3,albania,tirana,pronet-data-center,NaN
14,https://www.datacentermap.com/albania/tirana/r...,3,albania,tirana,rash-data-center,NaN
17,https://www.datacentermap.com/algeria/algiers/...,3,algeria,algiers,ayrade-dc-1--tassili-,NaN
18,https://www.datacentermap.com/algeria/algiers/...,3,algeria,algiers,hostarts-al01,NaN
19,https://www.datacentermap.com/algeria/algiers/...,3,algeria,algiers,huawei-mohammadia,NaN


### Parse structured fields from the URL hierarchy.

For a global dataset, `State` is too US-specific. This notebook keeps a more flexible structure:
- `Country`
- `Region Hierarchy` (everything between country and city)
- `City`
- `Datacenter Name`

In [8]:
def parse_segments(segments):
    """Parse a global DataCenterMap URL path into flexible geographic fields."""
    if not segments:
        return pd.Series({
            "Country": None,
            "Region Hierarchy": None,
            "City": None,
            "Datacenter Name": None
        })

    country = segments[0]
    datacenter_name = segments[-1] if len(segments) >= 1 else None
    city = segments[-2] if len(segments) >= 2 else None

    # Everything between country and city becomes regional hierarchy
    if len(segments) > 3:
        region_hierarchy = "/".join(segments[1:-2])
    else:
        region_hierarchy = None

    return pd.Series({
        "Country": country,
        "Region Hierarchy": region_hierarchy,
        "City": city,
        "Datacenter Name": datacenter_name
    })

parsed = detail_df["segments"].apply(parse_segments)
detail_df = pd.concat([detail_df, parsed], axis=1)

# Optional US-only convenience field for backwards compatibility
detail_df["State"] = detail_df.apply(
    lambda row: row["segments"][1] if row["Country"] == "usa" and len(row["segments"]) >= 4 else None,
    axis=1
)

detail_df = detail_df[
    [
        "Datacenter URL",
        "Country",
        "Region Hierarchy",
        "State",
        "City",
        "Datacenter Name",
        "depth",
        "path",
        "is_leaf",
        "seg0",
        "seg1",
        "seg2",
        "seg3",
        "seg4",
        "seg5",
    ]
].sort_values(["Country", "path"]).reset_index(drop=True)

detail_df.head(20)


,Datacenter URL,Country,Region Hierarchy,State,City,Datacenter Name,depth,path,is_leaf,seg0,seg1,seg2,seg3,seg4,seg5
0,https://www.datacentermap.com/afghanistan/kabu...,afghanistan,NaN,NaN,kabul,acg-data-center,3,afghanistan/kabul/acg-data-center,True,afghanistan,kabul,acg-data-center,NaN,NaN,None
1,https://www.datacentermap.com/afghanistan/kabu...,afghanistan,NaN,NaN,kabul,alef-technology,3,afghanistan/kabul/alef-technology,True,afghanistan,kabul,alef-technology,NaN,NaN,None
2,https://www.datacentermap.com/albania/tirana/a...,albania,NaN,NaN,tirana,albania-data-center-adc,3,albania/tirana/albania-data-center-adc,True,albania,tirana,albania-data-center-adc,NaN,NaN,None
3,https://www.datacentermap.com/albania/tirana/h...,albania,NaN,NaN,tirana,hostal-shpk,3,albania/tirana/hostal-shpk,True,albania,tirana,hostal-shpk,NaN,NaN,None
4,https://www.datacentermap.com/albania/tirana/k...,albania,NaN,NaN,tirana,keminet-shpk,3,albania/tirana/keminet-shpk,True,albania,tirana,keminet-shpk,NaN,NaN,None
5,https://www.datacentermap.com/albania/tirana/p...,albania,NaN,NaN,tirana,pronet-data-center,3,albania/tirana/pronet-data-center,True,albania,tirana,pronet-data-center,NaN,NaN,None
6,https://www.datacentermap.com/albania/tirana/r...,albania,NaN,NaN,tirana,rash-data-center,3,albania/tirana/rash-data-center,True,albania,tirana,rash-data-center,NaN,NaN,None
7,https://www.datacentermap.com/algeria/algiers/...,algeria,NaN,NaN,algiers,ayrade-dc-1--tassili-,3,algeria/algiers/ayrade-dc-1--tassili-,True,algeria,algiers,ayrade-dc-1--tassili-,NaN,NaN,None
8,https://www.datacentermap.com/algeria/algiers/...,algeria,NaN,NaN,algiers,hostarts-al01,3,algeria/algiers/hostarts-al01,True,algeria,algiers,hostarts-al01,NaN,NaN,None
9,https://www.datacentermap.com/algeria/algiers/...,algeria,NaN,NaN,algiers,huawei-mohammadia,3,algeria/algiers/huawei-mohammadia,True,algeria,algiers,huawei-mohammadia,NaN,NaN,None


### Save the global detail-page dataset and inspect country coverage.

In [9]:
detail_df.to_csv(OUTPUT_ALL_DETAIL, index=False, encoding="utf-8")
print(f"[DONE] Saved global detail-page dataset to '{OUTPUT_ALL_DETAIL}' with {len(detail_df):,} rows.")

print("\nTop countries:")
display(detail_df["Country"].value_counts().head(30))


[DONE] Saved global detail-page dataset to '02_datacentermap_detail_pages_global.csv' with 13,518 rows.

Top countries:


Country
usa                5125
united-kingdom      619
germany             572
ixp                 504
china               387
france              380
canada              348
india               328
australia           308
japan               273
italy               240
brazil              219
spain               219
the-netherlands     211
indonesia           204
russia              195
ireland             146
switzerland         134
malaysia            131
sweden              124
finland             116
norway              107
hong-kong           103
poland              103
south-korea         103
turkey               97
denmark              92
singapore            83
israel               76
romania              72
Name: count, dtype: int64

### Manual QA: inspect a random sample before relying on the output.

Leaf-page logic is much better than a fixed-depth rule, but it is still a heuristic. Always inspect a sample of URLs from several countries.

In [10]:
detail_df.sample(min(25, len(detail_df)), random_state=42)[
    ["Datacenter URL", "Country", "Region Hierarchy", "State", "City", "Datacenter Name"]
]


,Datacenter URL,Country,Region Hierarchy,State,City,Datacenter Name
12780,https://www.datacentermap.com/usa/virginia/fre...,usa,virginia,virginia,fredericksburg,project-sisson-dc1
4751,https://www.datacentermap.com/ixp/g/valence/,ixp,NaN,NaN,g,valence
6930,https://www.datacentermap.com/sweden/gavle/mic...,sweden,NaN,NaN,gavle,microsoft-valbo-gvx25
4860,https://www.datacentermap.com/japan/osaka/cyru...,japan,NaN,NaN,osaka,cyrusone-kep-osk1
6433,https://www.datacentermap.com/singapore/singap...,singapore,NaN,NaN,singapore,pacnet-sgcs2
13176,https://www.datacentermap.com/usa/virginia/ste...,usa,virginia,virginia,sterling,powerhouse-pacific-building-c
2769,https://www.datacentermap.com/germany/frankfur...,germany,NaN,NaN,frankfurt,northc-frankfurt
6119,https://www.datacentermap.com/russia/moscow/3d...,russia,NaN,NaN,moscow,3data-mr4-data-center
457,https://www.datacentermap.com/bangladesh/dhaka...,bangladesh,NaN,NaN,dhaka,coloasia-dc1
9638,https://www.datacentermap.com/usa/illinois/chi...,usa,illinois,illinois,chicago,600-780-south-federal
